# Inshallah — Phase 1 extraction prototype
Run the four cells in order. Cell 4 pulls the three real examples per label from the corpus and saves them with their full source records. Cell 2 needs the Kaggle keys for SADA (🔑 Secrets: `KAGGLE_USERNAME`, `KAGGLE_KEY`).
Expected total: **530,662** occurrences (SFC-mini 519,971; SADA 9,655; Gulf 1,036).

### 1. Install

In [ ]:
!pip install -q camel-tools huggingface_hub kagglehub

### 2. Download the data

In [ ]:
import os, zipfile, shutil
from pathlib import Path
from huggingface_hub import hf_hub_download

DATA = Path("/content/data")
for d in ["sfc", "gulf", "sada"]:
    (DATA / d).mkdir(parents=True, exist_ok=True)

# Gulf tweets (one CSV)
hf_hub_download("AhmedSSabir/Gulf-Arabic-Tweets-2018-2020", "arab_gen_twitter.csv", repo_type="dataset",
                revision="b549bc3dc6e25ca2980695d082de2818ec9733f2", local_dir=DATA / "gulf")
print("Gulf: done")

# SFC-mini (63 text files, about 3 GB)
for i in range(63):
    hf_hub_download("faisalq/SFC-mini", f"{i}.txt", repo_type="dataset",
                    revision="cae145d641901698f068f806ab27d95fba042ab9", local_dir=DATA / "sfc")
print("SFC-mini: done")

# SADA (three CSV files from Kaggle, version 23)
try:
    from google.colab import userdata
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    import kagglehub
    for f in ["train.csv", "valid.csv", "test.csv"]:
        p = kagglehub.dataset_download("sdaiancai/sada2022/versions/23", path=f)
        if zipfile.is_zipfile(p):
            with zipfile.ZipFile(p) as z:
                with z.open([n for n in z.namelist() if n.endswith(f)][0]) as src, open(DATA / "sada" / f, "wb") as dst:
                    shutil.copyfileobj(src, dst)
        else:
            shutil.copyfile(p, DATA / "sada" / f)
    print("SADA: done")
except Exception as e:
    print("SADA not downloaded:", e)
    print("Upload train.csv, valid.csv and test.csv to /content/data/sada/ and run cell 3.")

### 3. Find every occurrence and count the spellings

In [ ]:
import re
import pandas as pd
from collections import Counter
from camel_tools.utils.dediac import dediac_ar

DIAC = re.compile(r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]")
LETTER = r"\u0621-\u064A\u0671-\u06D3"
FORMULA = re.compile(rf"(?:(?<![{LETTER}])|(?<=(?<![{LETTER}])[وف]))ان\s*شا+\s*ء?\s*ا?لل[هة]+(?![{LETTER}])")

QUICK = re.compile(r"ن[\s\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED\u0640]*ش[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED\u0640]*[اآأإٱ]")

def find_variants(text):
    """Search a normalised copy; return each spelling as it appears in the original text."""
    if not QUICK.search(text):      # fast skip for texts without the formula
        return []
    out, idx = [], []
    for i, ch in enumerate(text):
        if DIAC.match(ch) or ch == "\u0640":
            continue
        out.append("ا" if ch in "آأإٱ" else ch)
        idx.append(i)
    found = []
    for m in FORMULA.finditer("".join(out)):
        start, end = idx[m.start()], idx[m.end() - 1] + 1
        while end < len(text) and (DIAC.match(text[end]) or text[end] == "\u0640"):
            end += 1
        found.append(text[start:end])
    return found

assert dediac_ar("إِنْ شَاءَ اللَّهُ") == "إن شاء الله"          # CAMeL Tools works
assert find_variants("انشالله بجي وان شاء الله نلتقي") == ["انشالله", "ان شاء الله"]
assert find_variants("ما شاء الله") == []

counts, texts = Counter(), {}
def scan(source, ref, text):
    for v in find_variants(text):
        counts[(source, v)] += 1
        texts[(source, ref)] = text

for i in range(63):
    with open(DATA / "sfc" / f"{i}.txt", encoding="utf-8") as fh:
        for ln, line in enumerate(fh, 1):
            if line.strip():
                scan("sfc", f"{i}.txt:{ln}", line.rstrip("\n"))
gulf = pd.read_csv(DATA / "gulf" / "arab_gen_twitter.csv", dtype=str, keep_default_na=False)
for i, t in enumerate(gulf["text"]):
    scan("gulf", f"arab_gen_twitter.csv:{i}", t)
for f in ["train.csv", "valid.csv", "test.csv"]:
    sada = pd.read_csv(DATA / "sada" / f, dtype=str, keep_default_na=False)
    for i, t in enumerate(sada["GroundTruthText"]):
        scan("sada", f"{f}:{i}", t)

proto = pd.DataFrame([(s, v, c) for (s, v), c in counts.items()], columns=["source", "variant", "count"])
proto = proto.sort_values(["source", "count"], ascending=[True, False])
proto.to_csv("prototype_variant_counts.csv", index=False, encoding="utf-8-sig")
print(proto.groupby("source")["count"].sum())
print("Total:", proto["count"].sum(), "| distinct spellings:", proto["variant"].nunique())

### 4. Pull three real examples per label

In [ ]:
EXAMPLES = [   # label, source, record reference (file:line or file:row), example as written in the source
    ("COMMIT", "sada", "train.csv:184833", "ورايح ان شاء الله أجيب الدواء"),
    ("COMMIT", "sada", "train.csv:197199", "أنا أبلغه ان شاء الله"),
    ("COMMIT", "sfc", "24.txt:115470", "راح ارتبط فيه ان شاء الله"),
    ("WISH", "sada", "train.csv:227867", "يا رب إن شاء الله يوفقك يا وليدي ويسعدك"),
    ("WISH", "sfc", "24.txt:94381", "ربي يوفقك ويسهل لك دربك وامرك ان شاء الله"),
    ("WISH", "sfc", "52.txt:111424", "وان شاء الله تتيسر اموركـ"),
    ("DEFER", "sada", "train.csv:191417", "أنا دارسها ومرتب لها إن شاء الله إن شاء الله محلولة"),
    ("DEFER", "sada", "train.csv:215316", "أنا إن شاء الله بعدين افهمك"),
    ("DEFER", "sada", "train.csv:131578", "ايه ان شاء الله ان شاء الله"),
    ("HEDGE", "sfc", "26.txt:10442", "ان شاء الله يكون وضحت لك"),
    ("HEDGE", "sada", "test.csv:6162", "إن شاء الله الحسابات طلعت مظبوطة يا عمي"),
    ("HEDGE", "sada", "train.csv:188757", "بس إن شاء الله ما يكون فات ألأاون"),
    ("SKEPTIC", "sfc", "14.txt:54146", "ان شالله لما يسمحوا لنا أوريكم الفلة على أصولها <<<< تحلم ههههه"),
    ("SKEPTIC", "sfc", "51.txt:121424", "متى كنت ناوي تبيع ان شاء الله"),
    ("SKEPTIC", "sada", "train.csv:227627", "وتبغينا نهرب و فين نروح إن شاء الله"),
    ("CLOSE", "sada", "train.csv:203201", "طيب ان شاء الله."),
    ("CLOSE", "sfc", "54.txt:61390", "ان شاء الله شكرا للمرور"),
    ("CLOSE", "sfc", "49.txt:17367", "فإلى الغد ان شاء الله"),
]
squash = lambda t: " ".join(t.split())
rows = []
for label, source, ref, example in EXAMPLES:
    record = texts.get((source, ref), "")
    found = squash(example) in squash(record)
    rows.append([label, source, ref, example, found, record])
    print("OK " if found else "NOT FOUND", label, source, ref, "|", example)
ex = pd.DataFrame(rows, columns=["label", "source", "record_ref", "example", "found_in_source", "full_record"])
ex.to_csv("label_examples_check.csv", index=False, encoding="utf-8-sig")
print("\nFound:", ex["found_in_source"].sum(), "of", len(ex))

from google.colab import files
files.download("prototype_variant_counts.csv")
files.download("label_examples_check.csv")